# Computation Graph Analysis

Builds computation graphs showing dependencies between components,
dataflow analysis, and computation cost per component.

**References:**
- Vig (2019) "A Multiscale Visualization of Attention in the Transformer Model"
- Conmy et al. (2023) "Towards Automated Circuit Discovery"

## Why This Matters

Computation graph analysis makes the implicit dataflow of a transformer explicit — tracing dependencies between components, identifying critical paths, and measuring interaction strengths. This provides a structural view of how information flows through the network.

**Key references:**
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.computation_graph import (
    component_dependency_graph,
    dataflow_analysis,
    computation_cost_profile,
    critical_path_analysis,
    component_interaction_strength,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads, d_model={cfg.d_model}")

In [ ]:
# 1. Component dependency graph
dep = component_dependency_graph(model, tokens, metric_fn)
print(f"Components: {len(dep['component_names'])}")
print(f"Edges found: {dep['n_edges']}")
if dep['edges']:
    print(f"\nDependency edges:")
    for src, tgt, w in dep['edges'][:10]:
        print(f"  {src} -> {tgt} (weight={w:.4f})")
    print(f"\nHighest out-degree:")
    sorted_out = sorted(dep['out_degree'].items(), key=lambda x: -x[1])[:5]
    for name, deg in sorted_out:
        if deg > 0:
            print(f"  {name}: {deg}")

In [ ]:
# 2. Dataflow analysis - information throughput
flow = dataflow_analysis(model, tokens)
print(f"Residual stream norms:")
for l in range(cfg.n_layers + 1):
    bar = '#' * int(flow['residual_norms'][l] * 5)
    print(f"  Layer {l}: {flow['residual_norms'][l]:.4f} {bar}")
print(f"\nAttn vs MLP throughput fractions:")
for l in range(cfg.n_layers):
    a_bar = '#' * int(flow['attn_fraction'][l] * 30)
    m_bar = '#' * int(flow['mlp_fraction'][l] * 30)
    print(f"  Layer {l}: attn={flow['attn_fraction'][l]:.3f} {a_bar}")
    print(f"          mlp={flow['mlp_fraction'][l]:.3f} {m_bar}")
print(f"\nPer-head attention throughput:")
for l in range(cfg.n_layers):
    vals = ' '.join(f"{flow['attn_throughput'][l,h]:.3f}" for h in range(cfg.n_heads))
    print(f"  Layer {l}: {vals}")

In [ ]:
# 3. Computation cost profile - effect per parameter
cost = computation_cost_profile(model, tokens, metric_fn)
print(f"Most cost-effective component: {cost['most_cost_effective']}")
print(f"\nPer-layer effects:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: attn_effect={cost['attn_effects'][l]:.4f} ({cost['attn_params'][l]:.0f} params), "
          f"mlp_effect={cost['mlp_effects'][l]:.4f} ({cost['mlp_params'][l]:.0f} params)")
print(f"\nCost-effectiveness (effect/param):")
sorted_ce = sorted(cost['cost_effectiveness'].items(), key=lambda x: -x[1])
for name, ce in sorted_ce:
    bar = '#' * int(ce * 1e6)
    print(f"  {name}: {ce:.2e} {bar}")

In [ ]:
# 4. Critical path analysis
path = critical_path_analysis(model, tokens, metric_fn)
print(f"Critical path ({path['path_length']} steps):")
print(f"Total effect: {path['total_path_effect']:.4f}")
for i, (comp, eff) in enumerate(zip(path['critical_path'], path['path_effects'])):
    bar = '#' * int(eff * 20 / (max(path['path_effects']) + 1e-10))
    print(f"  Step {i}: {comp} (effect={eff:.4f}) {bar}")

In [ ]:
# 5. Component interaction strength between layers
inter = component_interaction_strength(model, tokens, metric_fn, layer_a=0, layer_b=1)
print(f"Strongest interaction: {inter['strongest_interaction']}")
print(f"Mean interaction: {inter['mean_interaction']:.4f}")
n_comp = cfg.n_heads + 1
names_a = [f"H{h}" for h in range(cfg.n_heads)] + ["MLP"]
names_b = [f"H{h}" for h in range(cfg.n_heads)] + ["MLP"]
print(f"\nInteraction matrix (L0 x L1):")
header = '       ' + ' '.join(f"{n:>6}" for n in names_b)
print(header)
for i in range(n_comp):
    vals = ' '.join(f"{inter['interaction_matrix'][i,j]:6.3f}" for j in range(n_comp))
    print(f"  {names_a[i]:>4}: {vals}")